# Cost Decomposition Explorer

Adjust the sliders to see how the linear coefficient `a` (FFN/projection term) and quadratic coefficient `b` (attention term) shape the workspace cost curve. The crossover point S\* = a/b moves as you drag.

In [ ]:
# Copyright (c) 2026 J. Patrick Fulton
# Apache-2.0

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from visuals.common import A_FIT, B_FIT, A_CONS, B_CONS, MAX_SEQ, C_BLUE, C_RED, C_GREEN, C_GREY

# Enable interactive backend if ipympl is available; fall back to Agg silently.
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    _INTERACTIVE = True
except Exception:
    matplotlib.use("Agg")
    _INTERACTIVE = False

## Controls

- **a** — linear coefficient (bytes per token-position): captures FFN intermediates and projection matrices
- **b** — quadratic coefficient (bytes per token-position²): captures attention score tensors
- Use the preset buttons to load known configurations

In [ ]:
# Sliders
a_slider = widgets.FloatSlider(
    value=A_FIT, min=4096, max=65536, step=512,
    description="a (b/tok):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px"),
    readout_format=".0f",
)
b_slider = widgets.FloatSlider(
    value=B_FIT, min=0.1, max=30.0, step=0.1,
    description="b (b/tok²):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px"),
    readout_format=".1f",
)

# Preset buttons
btn_fitted = widgets.Button(description="Fitted (a=18432, b=6.2)", button_style="success")
btn_cons = widgets.Button(description="Conservative defaults", button_style="warning")

# Output widget for fallback mode (unused in interactive mode but defined always)
_out = widgets.Output()

# Create figure once for interactive mode; DO NOT call plt.show() — ipympl renders inline automatically
if _INTERACTIVE:
    _fig, (_ax1, _ax2) = plt.subplots(1, 2, figsize=(10, 5))


def update(a, b):
    S = np.linspace(1, MAX_SEQ, 2000)
    linear = a * S / 1e6
    quad = b * S**2 / 1e6
    total = linear + quad
    crossover = a / b

    if _INTERACTIVE:
        _ax1.cla()
        _ax2.cla()
        _fig.suptitle(
            f"Workspace Cost  (crossover S* = a/b = {crossover:,.0f} tokens)",
            fontsize=13,
        )
        _ax1.plot(S, linear, color=C_BLUE, lw=2, label=f"a·S (linear)  a={a:.0f}")
        _ax1.plot(S, quad, color=C_RED, lw=2, label=f"b·S² (quad)  b={b:.1f}")
        _ax1.plot(S, total, color=C_GREEN, lw=2.5, ls="--", label="Total W(S)")
        _ax1.axvline(crossover, color=C_GREY, lw=1.2, ls=":")
        _ax1.set_xlabel("Sequence length S (tokens)")
        _ax1.set_ylabel("Workspace (MiB)")
        _ax1.set_title("Linear axes")
        _ax1.legend(fontsize=9)
        _ax1.set_xlim(0, MAX_SEQ)
        _ax2.loglog(S, linear, color=C_BLUE, lw=2, label="slope 1")
        _ax2.loglog(S, quad, color=C_RED, lw=2, label="slope 2")
        _ax2.loglog(S, total, color=C_GREEN, lw=2.5, ls="--")
        _ax2.axvline(crossover, color=C_GREY, lw=1.2, ls=":")
        _ax2.set_xlabel("S (log scale)")
        _ax2.set_ylabel("Workspace bytes (log scale)")
        _ax2.set_title("Log-log axes")
        _ax2.legend(fontsize=9)
        _fig.tight_layout()
        _fig.canvas.draw_idle()
    else:
        with _out:
            _out.clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
            fig.suptitle(
                f"Workspace Cost  (crossover S* = a/b = {crossover:,.0f} tokens)",
                fontsize=13,
            )
            ax1.plot(S, linear, color=C_BLUE, lw=2, label=f"a·S (linear)  a={a:.0f}")
            ax1.plot(S, quad, color=C_RED, lw=2, label=f"b·S² (quad)  b={b:.1f}")
            ax1.plot(S, total, color=C_GREEN, lw=2.5, ls="--", label="Total W(S)")
            ax1.axvline(crossover, color=C_GREY, lw=1.2, ls=":")
            ax1.set_xlabel("Sequence length S (tokens)")
            ax1.set_ylabel("Workspace (MiB)")
            ax1.set_title("Linear axes")
            ax1.legend(fontsize=9)
            ax1.set_xlim(0, MAX_SEQ)
            ax2.loglog(S, linear, color=C_BLUE, lw=2, label="slope 1")
            ax2.loglog(S, quad, color=C_RED, lw=2, label="slope 2")
            ax2.loglog(S, total, color=C_GREEN, lw=2.5, ls="--")
            ax2.axvline(crossover, color=C_GREY, lw=1.2, ls=":")
            ax2.set_xlabel("S (log scale)")
            ax2.set_ylabel("Workspace bytes (log scale)")
            ax2.set_title("Log-log axes")
            ax2.legend(fontsize=9)
            fig.tight_layout()
            display(fig)
            plt.close(fig)


def on_change(change):
    update(a_slider.value, b_slider.value)


a_slider.observe(on_change, names="value")
b_slider.observe(on_change, names="value")


def on_fitted(_):
    a_slider.value = A_FIT
    b_slider.value = B_FIT


def on_cons(_):
    a_slider.value = A_CONS
    b_slider.value = B_CONS


btn_fitted.on_click(on_fitted)
btn_cons.on_click(on_cons)

controls = widgets.VBox([
    widgets.HBox([a_slider, b_slider]),
    widgets.HBox([btn_fitted, btn_cons]),
])
display(controls, _out)
update(A_FIT, B_FIT)

## What to look for

- At S < S\*, the blue line (FFN) dominates — a linear knob (static batch size) works here
- At S > S\*, the red line (attention) dominates — workspace grows quadratically and a static batch ceiling catastrophically under-budgets
- Try dragging `a` up and watch S\* move right (linear term becomes dominant at longer sequences)
- The conservative defaults (`a=16384, b=8`) vs fitted values shows the probe's improvement